In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

In [8]:
PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

WEATHER_FILE = (
    PROCESSED_DIR /
    "weather_daily_era5_cleaned.csv"
)

print("Weather file:", WEATHER_FILE)

Weather file: c:\Users\ASUS\Documents\GitHub\RiskWeave\data\processed\weather_daily_era5_cleaned.csv


In [9]:
weather = pd.read_csv(WEATHER_FILE)

weather["date"] = pd.to_datetime(weather["date"])

print("Dataset shape:", weather.shape)
print("Number of districts:", weather["district"].nunique())
print("First date:", weather["date"].min())
print("Last date:", weather["date"].max())

weather.head()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\ASUS\\Documents\\GitHub\\RiskWeave\\data\\processed\\weather_daily_era5_cleaned.csv'

In [6]:
weather.groupby("district").agg(
    total_records=("date", "count"),
    unique_dates=("date", "nunique"),
    first_date=("date", "min"),
    last_date=("date", "max")
)

NameError: name 'weather' is not defined

In [5]:
reporting_anchors = {
    2025: pd.Timestamp("2024-12-30"),
    2026: pd.Timestamp("2025-12-29")
}

for year, anchor in reporting_anchors.items():
    print(
        f"{year} Week 01: "
        f"{anchor.date()} to "
        f"{(anchor + pd.Timedelta(days=6)).date()}"
    )

2025 Week 01: 2024-12-30 to 2025-01-05
2026 Week 01: 2025-12-29 to 2026-01-04


In [8]:
calendar_records = []

for year, anchor in reporting_anchors.items():

    next_anchor = reporting_anchors.get(year + 1)

    if next_anchor is not None:
        period_end = next_anchor - pd.Timedelta(days=1)
    else:
        period_end = weather["date"].max()

    week_start = anchor
    week_number = 1

    while week_start <= period_end:

        week_end = week_start + pd.Timedelta(days=6)

        calendar_records.append({
            "reporting_year": year,
            "week_number": week_number,
            "week_start": week_start,
            "week_end": week_end
        })

        week_start += pd.Timedelta(days=7)
        week_number += 1

weekly_calendar = pd.DataFrame(calendar_records)

weekly_calendar.head(10)

,reporting_year,week_number,week_start,week_end
0,2025,1,2024-12-30,2025-01-05
1,2025,2,2025-01-06,2025-01-12
2,2025,3,2025-01-13,2025-01-19
3,2025,4,2025-01-20,2025-01-26
4,2025,5,2025-01-27,2025-02-02
5,2025,6,2025-02-03,2025-02-09
6,2025,7,2025-02-10,2025-02-16
7,2025,8,2025-02-17,2025-02-23
8,2025,9,2025-02-24,2025-03-02
9,2025,10,2025-03-03,2025-03-09


In [9]:
print("Total reporting weeks:", len(weekly_calendar))

print("\nFirst reporting weeks:")
display(weekly_calendar.head())

print("\nLast reporting weeks:")
display(weekly_calendar.tail())

Total reporting weeks: 96

First reporting weeks:


,reporting_year,week_number,week_start,week_end
0,2025,1,2024-12-30,2025-01-05
1,2025,2,2025-01-06,2025-01-12
2,2025,3,2025-01-13,2025-01-19
3,2025,4,2025-01-20,2025-01-26
4,2025,5,2025-01-27,2025-02-02



Last reporting weeks:


,reporting_year,week_number,week_start,week_end
91,2026,40,2026-09-28,2026-10-04
92,2026,41,2026-10-05,2026-10-11
93,2026,42,2026-10-12,2026-10-18
94,2026,43,2026-10-19,2026-10-25
95,2026,44,2026-10-26,2026-11-01


In [10]:
# Keep reporting weeks that end on or before
# the last available weather date

last_weather_date = weather["date"].max()

weekly_calendar = weekly_calendar[
    weekly_calendar["week_end"] <= last_weather_date
].copy()

weekly_calendar = weekly_calendar.reset_index(drop=True)

print("Complete reporting weeks:", len(weekly_calendar))

print("Last complete week:")
display(weekly_calendar.tail(1))

Complete reporting weeks: 95
Last complete week:


,reporting_year,week_number,week_start,week_end
94,2026,43,2026-10-19,2026-10-25


In [11]:
weekly_records = []

for _, week in weekly_calendar.iterrows():

    start_date = week["week_start"]
    end_date = week["week_end"]

    for district in weather["district"].unique():

        daily_records = weather[
            (weather["district"] == district) &
            (weather["date"] >= start_date) &
            (weather["date"] <= end_date)
        ]

        # Accept only weeks with seven daily observations
        if len(daily_records) != 7:
            continue

        weekly_records.append({

            "district": district,

            "reporting_year": week["reporting_year"],
            "week_number": week["week_number"],

            "week_start": start_date,
            "week_end": end_date,

            "weekly_rainfall_mm":
                daily_records["precipitation_sum"].sum(),

            "weekly_mean_temperature":
                daily_records["temperature_2m_mean"].mean(),

            "weekly_max_temperature":
                daily_records["temperature_2m_max"].max(),

            "weekly_min_temperature":
                daily_records["temperature_2m_min"].min(),

            "daily_observation_count": len(daily_records)
        })


weekly_weather = pd.DataFrame(weekly_records)

weekly_weather.head()

,district,reporting_year,week_number,week_start,week_end,weekly_rainfall_mm,weekly_mean_temperature,weekly_max_temperature,weekly_min_temperature,daily_observation_count
0,Ampara,2025,1,2024-12-30,2025-01-05,59.9,25.442857,28.5,23.6,7
1,Anuradhapura,2025,1,2024-12-30,2025-01-05,17.0,25.371429,29.8,21.3,7
2,Badulla,2025,1,2024-12-30,2025-01-05,63.7,21.600000,26.0,19.1,7
3,Batticaloa,2025,1,2024-12-30,2025-01-05,62.1,25.628571,29.0,22.4,7
4,Colombo,2025,1,2024-12-30,2025-01-05,56.9,26.385714,31.6,22.9,7


In [12]:
print("Weekly dataset shape:", weekly_weather.shape)

print(
    "Number of districts:",
    weekly_weather["district"].nunique()
)

print(
    "Number of reporting weeks:",
    weekly_weather[
        ["reporting_year", "week_number"]
    ].drop_duplicates().shape[0]
)

print(
    "Missing values:\n",
    weekly_weather.isna().sum()
)

print(
    "Daily observation counts:\n",
    weekly_weather["daily_observation_count"].value_counts()
)

Weekly dataset shape: (2375, 10)
Number of districts: 25
Number of reporting weeks: 95
Missing values:
 district                   0
reporting_year             0
week_number                0
week_start                 0
week_end                   0
weekly_rainfall_mm         0
weekly_mean_temperature    0
weekly_max_temperature     0
weekly_min_temperature     0
daily_observation_count    0
dtype: int64
Daily observation counts:
 daily_observation_count
7    2375
Name: count, dtype: int64


In [13]:
output_file = (
    PROCESSED_DIR / "weather_weekly_era5.csv"
)

weekly_weather.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Weekly weather data saved:")
print(output_file)

Weekly weather data saved:
c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\processed\weather_weekly_era5.csv
